# 03 Generate Clean CoT with Groq

Generate clean reasoning outputs for one prompt shard using Groq. This notebook keeps checkpoint/resume logic and writes failed records instead of crashing.


This cell installs and imports the Groq client dependencies plus utilities for JSONL, checkpointing, and progress display.

In [ ]:
%pip install -q groq pandas tqdm python-dotenv

from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import groq
from groq import Groq


This cell configures the Groq model, shard selection, checkpoint paths, final output path, and API key loading.

In [ ]:
# Main settings to edit before a run.
MODEL_NAME = "llama-3.3-70b-versatile"
SHARD_ID = 1       # 1-based: choose the contiguous shard owned by this machine.
NUM_SHARDS = 2
CHECKPOINT_INTERVAL = 10
MAX_SAMPLES = None # Use a small number for testing, or None for the full shard.
RETRY_FAILED = False

TEMPERATURE = 0.2
TOP_P = 0.9
MAX_TOKENS = 700

BACKEND = "groq"
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
# CHANGED: generated/intermediate outputs stay on E: to avoid filling the C: drive.
OUTPUT_ROOT = Path(r"E:\PFE_CoT_outputs")
PROMPT_DATA_PATH = OUTPUT_ROOT / "data" / "interim" / f"reasoning_prompts_shard{SHARD_ID}.jsonl"
CHECKPOINT_DIR = OUTPUT_ROOT / "data" / "checkpoint"
FINAL_DIR = OUTPUT_ROOT / "data" / "final"


def model_slug(model_name):
    aliases = {
        "llama-3.3-70b-versatile": "llama33",
    }
    if model_name in aliases:
        return aliases[model_name]
    return re.sub(r"[^A-Za-z0-9]+", "", model_name).lower()


MODEL_SAFE_NAME = model_slug(MODEL_NAME)
RUN_NAME = f"{BACKEND}_{MODEL_SAFE_NAME}_shard{SHARD_ID}"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_checkpoint.jsonl"
FINAL_OUTPUT_PATH = FINAL_DIR / f"{RUN_NAME}.jsonl"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

if SHARD_ID < 1 or SHARD_ID > NUM_SHARDS:
    raise ValueError("SHARD_ID must be between 1 and NUM_SHARDS.")
if not PROMPT_DATA_PATH.exists():
    raise FileNotFoundError(f"Run notebook 02 for SHARD_ID={SHARD_ID} first. Missing prompt shard: {PROMPT_DATA_PATH}")

load_dotenv(PROJECT_ROOT / ".env")
raw_keys = os.getenv("GROQ_API_KEYS") or os.getenv("GROQ_API_KEY", "")
GROQ_API_KEYS = [key.strip() for key in raw_keys.split(",") if key.strip()]
if not GROQ_API_KEYS:
    raise RuntimeError("Set GROQ_API_KEY or GROQ_API_KEYS in .env before running Groq inference.")

print("Prompt shard:", PROMPT_DATA_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Final output:", FINAL_OUTPUT_PATH)
print("Groq keys loaded:", len(GROQ_API_KEYS))


This cell loads the assigned prompt shard, verifies that `original_id` is present, and prints a quick category summary.

In [ ]:
def load_jsonl(path):
    records = []
    if not path.exists():
        return records
    with open(path, "r", encoding="utf-8") as input_handle:
        for line in input_handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


prompt_records = load_jsonl(PROMPT_DATA_PATH)
missing_original_id = [record for record in prompt_records if not record.get("original_id")]
if missing_original_id:
    raise KeyError("Prompt records must contain original_id. Re-run notebook 02 after updating the pipeline.")

if MAX_SAMPLES is not None:
    prompt_records = prompt_records[:MAX_SAMPLES]

print("Prompt records in this shard:", len(prompt_records))
if prompt_records:
    prompt_df = pd.DataFrame(prompt_records)
    print("Unique original_id values:", prompt_df["original_id"].nunique())
    print("Category distribution:")
    print(prompt_df["category"].value_counts(dropna=False).head(30))


This cell defines verdict parsing, label comparison, and output-record creation with up to three genuine model attempts per sample.

In [ ]:
MAX_GENERATION_ATTEMPTS = 3


def clean_label(value):
    text = "unknown" if value is None else str(value).strip()
    return text if text else "unknown"


def normalize_for_compare(value):
    text = clean_label(value).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .;:,_`*")


def parse_verdict(text):
    if not isinstance(text, str) or text == "failed_generation":
        return "failed_generation" if text == "failed_generation" else "unknown"
    match = re.search(r"(?im)^\s*Verdict\s*:\s*(?:\n\s*)?([^\r\n]+)", text)
    if match:
        return match.group(1).strip().strip(" .;:,_`*")
    return "unknown"


def make_output_record(prompt_record):
    sample_id = int(prompt_record["sample_id"])
    original_id = str(prompt_record["original_id"])
    true_label = clean_label(prompt_record.get("true_label", "unknown"))
    expected = normalize_for_compare(true_label)

    for attempt_number in range(1, MAX_GENERATION_ATTEMPTS + 1):
        try:
            reasoning_text = generate_with_groq(prompt_record)
            parsed_verdict = parse_verdict(reasoning_text)
        except Exception:
            reasoning_text = "failed_generation"
            parsed_verdict = "exception"

        parsed_normalized = normalize_for_compare(parsed_verdict)
        if parsed_normalized == expected:
            print(f"[OK] sample_id={sample_id} original_id={original_id} attempt={attempt_number} verdict={parsed_verdict}")
            return {
                "sample_id": sample_id,
                "original_id": original_id,
                "backend": BACKEND,
                "model": MODEL_NAME,
                "shard_id": SHARD_ID,
                "source_dataset": prompt_record.get("source_dataset", "CICMalDroid2020"),
                "prompt": prompt_record.get("prompt", ""),
                "system_prompt": prompt_record.get("system_prompt", ""),
                "user_prompt": prompt_record.get("user_prompt", prompt_record.get("prompt", "")),
                "raw_generation": reasoning_text,
                "reasoning": reasoning_text,
                "prediction": parsed_verdict,
            }

        if parsed_normalized in {"", "unknown", "exception", "failed_generation"}:
            print(f"[RETRY] sample_id={sample_id} original_id={original_id} attempt={attempt_number} no parseable verdict expected={true_label}")
        else:
            print(f"[MISMATCH] sample_id={sample_id} original_id={original_id} attempt={attempt_number} predicted={parsed_verdict} expected={true_label}")

    print(f"[FAILED] sample_id={sample_id} original_id={original_id} all 3 attempts failed verdict validation")
    return make_failed_record(
        prompt_record,
        RuntimeError("model failed to produce a matching parseable verdict in 3 attempts"),
    )


def make_failed_record(prompt_record, error):
    return {
        "sample_id": int(prompt_record["sample_id"]),
        "original_id": str(prompt_record["original_id"]),
        "backend": BACKEND,
        "model": MODEL_NAME,
        "shard_id": SHARD_ID,
        "source_dataset": prompt_record.get("source_dataset", "CICMalDroid2020"),
        "prompt": prompt_record.get("prompt", ""),
        "system_prompt": prompt_record.get("system_prompt", ""),
        "user_prompt": prompt_record.get("user_prompt", prompt_record.get("prompt", "")),
        "raw_generation": "failed_generation",
        "reasoning": "failed_generation",
        "prediction": "failed_generation",
        "error_message": str(error),
    }


This cell defines checkpoint and final-output helpers keyed by `original_id` so resume logic does not rely on temporary `sample_id` values.

In [ ]:
def record_key(record):
    if record.get("original_id"):
        return str(record["original_id"])
    return f"missing_original_id::sample_{record['sample_id']}"


def records_by_original_id(paths):
    completed = {}
    for path in paths:
        for record in load_jsonl(path):
            completed[record_key(record)] = record
    return completed


def save_checkpoint(completed):
    temp_path = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + ".tmp")
    ordered_records = sorted(completed.values(), key=lambda record: (str(record.get("original_id", "")), int(record["sample_id"])))
    with open(temp_path, "w", encoding="utf-8") as output_handle:
        for record in ordered_records:
            output_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temp_path.replace(CHECKPOINT_PATH)


def save_final_output(completed):
    temp_path = FINAL_OUTPUT_PATH.with_suffix(FINAL_OUTPUT_PATH.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as output_handle:
        for prompt_record in prompt_records:
            sample_key = str(prompt_record["original_id"])
            if sample_key in completed:
                output_handle.write(json.dumps(completed[sample_key], ensure_ascii=False) + "\n")
    temp_path.replace(FINAL_OUTPUT_PATH)


This cell creates the Groq client and defines the actual Groq chat-completion call used by the retry loop.

In [ ]:
current_key_index = 0


def groq_client():
    return Groq(api_key=GROQ_API_KEYS[current_key_index])


MAX_GROQ_RATE_LIMIT_RETRIES = 5
INITIAL_GROQ_BACKOFF_SECONDS = 5
MAX_GROQ_BACKOFF_SECONDS = 60


def is_rate_limit_error(error):
    status_code = getattr(error, "status_code", None)
    response = getattr(error, "response", None)
    response_status = getattr(response, "status_code", None)
    return isinstance(error, groq.RateLimitError) or status_code == 429 or response_status == 429


def generate_with_groq(prompt_record):
    system_prompt = prompt_record.get("system_prompt", "You are a malware-analysis reasoning assistant.")
    user_prompt = prompt_record.get("user_prompt", prompt_record.get("prompt", ""))
    backoff_seconds = INITIAL_GROQ_BACKOFF_SECONDS
    for retry_number in range(MAX_GROQ_RATE_LIMIT_RETRIES + 1):
        try:
            response = groq_client().chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()
        except Exception as error:
            if not is_rate_limit_error(error) or retry_number >= MAX_GROQ_RATE_LIMIT_RETRIES:
                raise
            print(f"[RATE_LIMIT] Groq 429; sleeping {backoff_seconds}s before retry {retry_number + 1}/{MAX_GROQ_RATE_LIMIT_RETRIES}")
            time.sleep(backoff_seconds)
            backoff_seconds = min(backoff_seconds * 2, MAX_GROQ_BACKOFF_SECONDS)


This cell resumes completed generations, processes unfinished prompt records, and periodically saves checkpoint and final JSONL outputs.

In [ ]:
completed = records_by_original_id([FINAL_OUTPUT_PATH, CHECKPOINT_PATH])
print("Records already completed:", len(completed))

processed_since_save = 0
new_records = 0
for prompt_record in tqdm(prompt_records, total=len(prompt_records)):
    sample_key = str(prompt_record["original_id"])
    already_done = sample_key in completed and completed[sample_key].get("prediction") != "failed_generation"
    failed_done = sample_key in completed and completed[sample_key].get("prediction") == "failed_generation"
    if already_done or (failed_done and not RETRY_FAILED):
        continue

    try:
        completed[sample_key] = make_output_record(prompt_record)
    except Exception as error:
        print(f"Generation failed for original_id={sample_key}, sample_id={prompt_record['sample_id']}: {error}")
        completed[sample_key] = make_failed_record(prompt_record, error)

    processed_since_save += 1
    new_records += 1
    if processed_since_save >= CHECKPOINT_INTERVAL:
        save_checkpoint(completed)
        save_final_output(completed)
        processed_since_save = 0
        time.sleep(0.2)

if new_records or not CHECKPOINT_PATH.exists():
    save_checkpoint(completed)
    print("Checkpoint saved:", CHECKPOINT_PATH)
if new_records or not FINAL_OUTPUT_PATH.exists():
    save_final_output(completed)
    print("Final output saved:", FINAL_OUTPUT_PATH)
if not new_records and FINAL_OUTPUT_PATH.exists():
    print("No new records generated; existing final output left unchanged.")


This cell performs a quick final-output sanity check and previews parsed predictions from the generated file.

In [ ]:
# Quick sanity check.
final_records = load_jsonl(FINAL_OUTPUT_PATH)
print("Final records:", len(final_records))
if final_records:
    result_df = pd.DataFrame(final_records)
    print(result_df["prediction"].value_counts(dropna=False).head(30))
    display(result_df[["sample_id", "original_id", "backend", "model", "shard_id", "prediction"]].head())
